In [ ]:
# 채팅 모델에서 few shot 사용

In [ ]:
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain.memory import ChatMessageHistory
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
# FewShotChatMessagePromptTemplate

import os
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate
)
from langchain_core.messages import HumanMessage, AIMessage
    
examples = [
    {
        "input": HumanMessage(content="안녕하세요, 오늘 날씨가 좋네요."),
        "output": AIMessage(content="네, 정말 기분 좋은 날씨네요!"),
    },
    {
        "input": HumanMessage(content="아, 배고파. 오늘 뭐 먹지?"),
        "output": AIMessage(content="맛있는 점심 메뉴를 추천해 드릴까요?"),
    },
]

# from_messages는 전체 형태를 정의
# format_messages는 값을 채우는 역할
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

# FewShotChatMessagePromptTemplate 객체 생성
# examples는 예시 데이터 목록
# example_prompt는 examples를 어떤 형태로 전달할지 정의
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 사용자에게 항상 친절하고 공감하는 챗봇이야."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
chain = final_prompt | llm

response = chain.invoke({"input": "이번주도 벌써 반이 지나갔네."})
print(response.content)

In [ ]:
# FewShotChatMessagePromptTemplate + RunnableWithMessageHistory

import os
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate
)
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

examples = [
    {
        "input": HumanMessage(content="오늘 날씨가 어때?"),
        "output": AIMessage(content="네, 오늘 날씨는 맑고 따뜻합니다."),
    },
    {
        "input": HumanMessage(content="커피 한 잔 마시고 싶다."),
        "output": AIMessage(content="네, 근처에 맛있는 카페를 찾아볼까요?"),
    },
]

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# 'history'와 'few_shot_prompt'가 모두 포함됩니다.
final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 사용자에게 항상 친절하고 '네'로 시작하는 인공지능 비서야."),
        few_shot_prompt, # ("human", "{input}"), ("ai", "{output}"),
        ("placeholder", "{history}"), # MessagesPlaceholder(variable_name="history")
        ("human", "{input}"),
    ]
)

store = {}
def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 6. 체인 생성 (프롬프트 + LLM) 및 메모리 관리 객체 결합
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
chain = final_prompt | llm
with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
    # {'input': ■1, 'history': ■2}
)

session_id = "user_abc"

response1 = with_history.invoke(
    {"input": "나의 이름은 제이슨이야."},
    config={"configurable": {"session_id": session_id}}
)
print(f"첫 번째 응답 (페르소나 적용): {response1.content}\n")

response2 = with_history.invoke(
    {"input": "내가 방금 뭐라고 말했었지?"},
    config={"configurable": {"session_id": session_id}}
)
print(f"두 번째 응답 (기억 + 페르소나): {response2.content}")

In [ ]:
# RunnableWithMessageHistory + FileChatMessageHistory

import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import FileChatMessageHistory



def get_session_history(session_id: str) -> FileChatMessageHistory:
    return FileChatMessageHistory(f"chat_histories/{session_id}.json")

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 사용자에게 항상 친절하고 도움이 되는 AI야."),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)

chain = RunnableWithMessageHistory(
    runnable=prompt | llm,
    get_session_history=get_session_history,
    input_messages_key="input",
    history_messages_key="history",
    # {'input': ■1, 'history': ■2}
)

print("--- 첫 번째 대화 ---")
response1 = chain.invoke(
    {"input": "안녕, 오늘 날씨가 참 좋네요."},
    config={"configurable": {"session_id": "my-session"}},
)
print(response1.content)

# 5. 동일한 세션으로 두 번째 대화
# 'my-session.json' 파일에 기록된 이전 대화를 자동으로 불러옴
print("\n--- 두 번째 대화 ---")
response2 = chain.invoke(
    {"input": "어제는 뭐했어?"},
    config={"configurable": {"session_id": "my-session"}},
)
print(response2.content)

# 새로운 세션으로 대화 시작
# 'another-session.json'이라는 새로운 파일이 생성됨
print("\n--- 새로운 세션으로 대화 ---")
response3 = chain.invoke(
    {"input": "오늘 점심 뭐 먹지?"},
    config={"configurable": {"session_id": "another-session"}},
)
print(response3.content)

In [ ]:
# RunnableBranch: 체인(runnable)
# ConditionalPromptSelector: 프롬프트

In [ ]:
# ConditionalPromptSelector

# 입력 값에 따라 프롬프트 템플릿을 동적으로 선택

# 주요 속성
# default_prompt: 어떤 조건에도 해당하지 않을 때 사용될 기본 프롬프트 템플릿

# conditionals: 
# [(callable, prompt),...] > callable의 리턴이 True이면 prompt를 선택, False이면 다음 조건으로 이동

In [ ]:
from langchain.chains.prompt_selector import ConditionalPromptSelector
from langchain.prompts import PromptTemplate

# 프롬프트 정의
ko_prompt = PromptTemplate.from_template("다음 문장을 한국어로 번역해 주세요: {text}")
en_prompt = PromptTemplate.from_template("Translate the following sentence to English: {text}")

# 조건 함수 정의
# {"language": "ko", "text": "Hello"}가 input_dict
# {"language": "en", "text": "안녕하세요"}가 input_dict
def is_korean(input_dict):
    return input_dict.get("language") == "ko"

# ConditionalPromptSelector 객체 생성 (올바른 사용법)
# 조건과 프롬프트를 튜플로 묶어서 전달
prompt_selector = ConditionalPromptSelector(
    default_prompt=en_prompt,
    conditionals=[(is_korean, ko_prompt)]
)

# 올바르게 동작하는 것을 확인
selected_prompt = prompt_selector.get_prompt({"language": "ko", "text": "Hello"})
print(selected_prompt.format(text="Hello"))

selected_prompt_en = prompt_selector.get_prompt({"language": "en", "text": "안녕하세요"})
print(selected_prompt_en.format(text="안녕하세요"))

In [ ]:
from langchain.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains.prompt_selector import ConditionalPromptSelector
from langchain_core.output_parsers import StrOutputParser

# 프롬프트 정의
default_prompt = PromptTemplate.from_template("다음 주제에 대해서 이야기 해봐 {topic}.")
chat_prompt = PromptTemplate.from_template("넌 도움이 많이 되는 AI 도우미야. 다음 주제에 대해서 이야기 해봐 {topic}.")

# 조건 함수
def is_gemini_flash(llm):
    return "gemini" in llm.model and "flash" in llm.model
    # llm 객체의 model 속성 문자열에 "gemini"와 "flash"가 모두 포함되어 있어야 한다는 조건

# 프롬프트 선택기
selector = ConditionalPromptSelector(
    default_prompt=default_prompt,
    conditionals=[(is_gemini_flash, chat_prompt)]
)

# 모델 인스턴스
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

# LCEL 체인 구성 및 실행
# selector.get_prompt(llm)에서 전달한 llm은
# conditionals의 리스트를 순회하면서 callable에 전달
chain = selector.get_prompt(llm) | llm | StrOutputParser()
response = chain.invoke({"topic": "black holes"})
print(response)
